# 🎬 G10 — Fine-tuning CamemBERT sur Allociné
## Problématique P02 : Régularisation & Généralisation

---

| Paramètre | Valeur |
|-----------|--------|
| **Dataset** | D05 — Allociné (100k critiques FR, 2 classes) |
| **Modèle** | M04 — CamemBERT-base (110M paramètres) |
| **Problématique** | P02 — Régularisation et Généralisation |
| **Méthode** | Optuna (Bayésien — TPE) |
| **Métrique** | F1-score macro |
| **Date limite** | 13 mars 2026 |

### Question de recherche
> **Comment le `weight_decay` et le `dropout` affectent-ils la généralisation de CamemBERT fine-tuné sur des critiques de films en français ?**

### Plan du notebook
0. Installation & imports  
1. Exploration du dataset Allociné  
2. Configuration du modèle CamemBERT  
3. Baseline (hyperparamètres par défaut)  
4. Grid search P02 (weight_decay × dropout)  
5. Optimisation Optuna (Bayésienne)  
6. Analyse du loss landscape & sharpness  
7. Entraînement final (meilleurs hyperparamètres)  
8. Visualisations & synthèse  

---

## 0. Installation & Imports

In [ ]:
# ── Installation (à lancer une seule fois) ─────────────────────────────────
# Si vous utilisez Poetry : poetry install  (dans le terminal)
# Sinon, décommentez la ligne ci-dessous :
# !pip install torch transformers datasets optuna scikit-learn matplotlib seaborn sentencepiece evaluate accelerate -q

In [ ]:
# ── Imports standards ──────────────────────────────────────────────────────
import os
import sys
import copy
import json
import time
import random
import logging
import warnings
import pickle
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field
from typing import Optional

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

# ── Data science ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import seaborn as sns

# ── Deep Learning ──────────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# ── HuggingFace ────────────────────────────────────────────────────────────
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    CamembertConfig,
    CamembertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ── Métriques ──────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

# ── Optuna ─────────────────────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Style matplotlib ───────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = sns.color_palette('husl', 8)
sns.set_theme(style='whitegrid')

# ── Dossier résultats ──────────────────────────────────────────────────────
RESULTS_DIR = Path('../results')
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('✅ Imports OK')
print(f'   PyTorch : {torch.__version__}')
print(f'   CUDA disponible : {torch.cuda.is_available()}')

---
## 1. Exploration du Dataset Allociné (D05)

In [ ]:
# ── 1.1 Chargement ────────────────────────────────────────────────────────
print('Chargement du dataset Allociné...')
raw_dataset = load_dataset('allocine')
print(raw_dataset)

In [ ]:
# ── 1.2 Statistiques descriptives ─────────────────────────────────────────
LABEL_NAMES = {0: 'Négatif 😠', 1: 'Positif 😊'}

for split_name, split in raw_dataset.items():
    label_counts = Counter([ex['label'] for ex in split])
    lengths = [len(ex['review'].split()) for ex in list(split)[:3000]]
    print(f'\n── {split_name} ({len(split):,} exemples) ──')
    for lbl, cnt in sorted(label_counts.items()):
        print(f'   {LABEL_NAMES[lbl]} : {cnt:,} ({cnt/len(split)*100:.1f}%)')
    print(f'   Longueur moy. : {np.mean(lengths):.0f} mots | max : {max(lengths)}')

In [ ]:
# ── 1.3 Exemples de critiques ─────────────────────────────────────────────
for label in [0, 1]:
    ex = next(e for e in raw_dataset['train'] if e['label'] == label)
    print(f'\n{"="*65}')
    print(f'  {LABEL_NAMES[label]}')
    print('='*65)
    print(f'  "{ex["review"][:350]}..."')

In [ ]:
# ── 1.4 Distribution des longueurs ────────────────────────────────────────
sample = list(raw_dataset['train'])[:5000]
lengths_neg = [len(e['review'].split()) for e in sample if e['label'] == 0]
lengths_pos = [len(e['review'].split()) for e in sample if e['label'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogramme par classe
axes[0].hist(lengths_neg, bins=50, alpha=0.7, color=PALETTE[0], label='Négatif', edgecolor='white')
axes[0].hist(lengths_pos, bins=50, alpha=0.7, color=PALETTE[3], label='Positif', edgecolor='white')
axes[0].axvline(256, color='red', lw=2, ls='--', label='Troncature (256)')
axes[0].set_xlabel('Longueur (mots)', fontsize=11)
axes[0].set_ylabel('Fréquence', fontsize=11)
axes[0].set_title('Distribution des longueurs par classe', fontsize=12)
axes[0].legend()

# Distribution cumulée
all_lengths = lengths_neg + lengths_pos
sorted_l = np.sort(all_lengths)
cum = np.arange(1, len(sorted_l)+1) / len(sorted_l)
axes[1].plot(sorted_l, cum * 100, color=PALETTE[2], lw=2.5)
axes[1].axvline(256, color='red', lw=2, ls='--', label=f'256 mots ({sum(1 for l in all_lengths if l<=256)/len(all_lengths)*100:.0f}% couvert)')
axes[1].set_xlabel('Longueur (mots)', fontsize=11)
axes[1].set_ylabel('% critiques couvertes', fontsize=11)
axes[1].set_title('Distribution cumulée des longueurs', fontsize=12)
axes[1].legend()

fig.suptitle('Analyse des longueurs — Dataset Allociné (D05)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'length_distribution.png', bbox_inches='tight')
plt.show()

trunc_pct = sum(1 for l in all_lengths if l > 256) / len(all_lengths) * 100
print(f'Critiques tronquées à 256 mots : {trunc_pct:.1f}%')

---
## 2. Préparation du Modèle CamemBERT

In [ ]:
# ── 2.1 Configuration globale ─────────────────────────────────────────────
MODEL_NAME    = 'camembert-base'
NUM_LABELS    = 2
MAX_SEQ_LEN   = 256
TEXT_COL      = 'review'
LABEL_COL     = 'label'

# Tailles des sous-ensembles (adapté CPU)
N_TRAIN_PER_CLASS = 500   # → 1 000 train
N_VAL_PER_CLASS   = 150   # → 300 val
N_TEST_PER_CLASS  = 150   # → 300 test

# Hyperparamètres à explorer (protocole P02)
WEIGHT_DECAY_GRID = [1e-5, 1e-4, 1e-3, 1e-2]
DROPOUT_GRID      = [0.0, 0.1, 0.3]

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    torch.set_num_threads(4)
print(f'Device : {DEVICE}')

In [ ]:
# ── 2.2 Tokenizer ─────────────────────────────────────────────────────────
print(f'Chargement du tokenizer {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, model_max_length=512)
print(f'Vocabulaire : {tokenizer.vocab_size:,} tokens')

# Exemple de tokenisation
exemple = raw_dataset['train'][0]['review'][:200]
tokens = tokenizer.tokenize(exemple)
print(f'\nExemple de tokenisation (SentencePiece) :')
print(f'  Texte  : "{exemple}"')
print(f'  Tokens : {tokens[:20]}...')
print(f'  Nombre : {len(tokens)} tokens')

In [ ]:
# ── 2.3 Sous-échantillonnage équilibré ────────────────────────────────────
def balanced_subsample(dataset_split, n_per_class, seed=SEED):
    """Retourne n_per_class exemples par classe (sous-ensemble équilibré)."""
    random.seed(seed)
    by_class = {}
    for ex in dataset_split:
        by_class.setdefault(ex[LABEL_COL], []).append(ex)
    result = []
    for label, exs in by_class.items():
        selected = random.sample(exs, min(n_per_class, len(exs)))
        result.extend(selected)
    random.shuffle(result)
    return result

train_samples = balanced_subsample(raw_dataset['train'],      N_TRAIN_PER_CLASS)
val_samples   = balanced_subsample(raw_dataset['validation'], N_VAL_PER_CLASS)
test_samples  = balanced_subsample(raw_dataset['test'],       N_TEST_PER_CLASS)

print(f'Sous-ensembles créés :')
print(f'  Train : {len(train_samples)} exemples ({N_TRAIN_PER_CLASS}/classe)')
print(f'  Val   : {len(val_samples)} exemples ({N_VAL_PER_CLASS}/classe)')
print(f'  Test  : {len(test_samples)} exemples ({N_TEST_PER_CLASS}/classe)')

In [ ]:
# ── 2.4 Dataset PyTorch ────────────────────────────────────────────────────
class AllocinéDataset(Dataset):
    """Dataset PyTorch pour Allociné tokenisé."""
    def __init__(self, samples, tokenizer, max_length=MAX_SEQ_LEN):
        texts  = [ex[TEXT_COL]  for ex in samples]
        labels = [ex[LABEL_COL] for ex in samples]
        enc = tokenizer(
            texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        )
        self.input_ids      = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels':         self.labels[idx],
        }

print('Tokenisation des données...')
train_ds = AllocinéDataset(train_samples, tokenizer)
val_ds   = AllocinéDataset(val_samples,   tokenizer)
test_ds  = AllocinéDataset(test_samples,  tokenizer)
print(f'✅ Datasets PyTorch créés — shape input_ids : {train_ds.input_ids.shape}')

In [ ]:
# ── 2.5 Fonction de chargement du modèle ──────────────────────────────────
def load_camembert(dropout=0.1, device=DEVICE):
    """
    Charge CamemBERT-base avec dropout configurable (requis par P02).
    Le dropout est appliqué :
      - aux couches cachées (hidden_dropout_prob)
      - aux couches d'attention (attention_probs_dropout_prob)
      - à la tête de classification (classifier_dropout)
    """
    config = CamembertConfig.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
        classifier_dropout=dropout,
    )
    model = CamembertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config,
        ignore_mismatched_sizes=True,
    )
    return model.to(device)

# Aperçu du modèle
tmp_model = load_camembert(dropout=0.1)
total_params = sum(p.numel() for p in tmp_model.parameters())
trainable    = sum(p.numel() for p in tmp_model.parameters() if p.requires_grad)
print(f'CamemBERT-base chargé :')
print(f'  Paramètres totaux     : {total_params/1e6:.1f} M')
print(f'  Paramètres entraîn.   : {trainable/1e6:.1f} M')
del tmp_model

---
## 3. Fonctions d'Entraînement & Métriques

In [ ]:
# ── 3.1 Métriques ─────────────────────────────────────────────────────────
def compute_metrics(labels, preds):
    """Calcule accuracy et F1-score macro."""
    labels = np.array(labels)
    preds  = np.array(preds)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1_macro': float(f1_score(labels, preds, average='macro', zero_division=0)),
    }

def generalization_gap(train_f1, val_f1):
    """Gap de généralisation (P02)."""
    gap = train_f1 - val_f1
    return {'train_f1': train_f1, 'val_f1': val_f1,
            'gap': gap, 'gap_pct': gap/train_f1*100 if train_f1 > 0 else 0.}

def compute_sharpness(base_loss, perturbed_losses):
    """Sharpness = (1/N) Σ |L(θ+εd_i) - L(θ)|  (formule du cours)."""
    return float(np.mean([abs(l - base_loss) for l in perturbed_losses]))

print('✅ Métriques définies')

In [ ]:
# ── 3.2 Boucle d'entraînement ─────────────────────────────────────────────
def train_model(
    model, train_dataset, val_dataset,
    lr=2e-5, weight_decay=1e-4,
    batch_size=16, grad_accum=2,
    num_epochs=3, warmup_ratio=0.1,
    early_stopping_patience=2,
    device=DEVICE, seed=SEED,
    verbose=True,
):
    """
    Entraîne CamemBERT avec :
      - AdamW + weight_decay (sans decay sur bias/LayerNorm)
      - Scheduler linéaire avec warmup
      - Gradient accumulation (simulation grands batches sur CPU)
      - Early stopping sur F1-val
    Retourne : history dict + meilleur F1 val
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size*2, shuffle=False, num_workers=0)

    # AdamW : exclusion des biais et LayerNorm du weight decay (standard Transformers)
    no_decay = ['bias', 'LayerNorm.weight']
    opt_params = [
        {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': weight_decay},
        {'params': [p for n,p in model.named_parameters() if     any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optimizer = AdamW(opt_params, lr=lr)

    steps_per_epoch = len(train_loader) // grad_accum
    total_steps     = steps_per_epoch * num_epochs
    warmup_steps    = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[], 'train_acc':[], 'val_acc':[]}
    best_val_f1 = -1.
    best_state  = None
    patience_ctr = 0
    t_start = time.time()

    for epoch in range(1, num_epochs+1):
        # ── Train ──
        model.train()
        train_loss, all_labels, all_preds = 0., [], []
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            batch   = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss    = outputs.loss / grad_accum
            loss.backward()
            train_loss += outputs.loss.item()
            all_preds.extend(outputs.logits.argmax(-1).cpu().tolist())
            all_labels.extend(batch['labels'].cpu().tolist())

            if (step+1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

        train_m = compute_metrics(all_labels, all_preds)
        avg_train_loss = train_loss / len(train_loader)

        # ── Val ──
        model.eval()
        val_loss, all_labels, all_preds = 0., [], []
        with torch.no_grad():
            for batch in val_loader:
                batch   = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                val_loss += outputs.loss.item()
                all_preds.extend(outputs.logits.argmax(-1).cpu().tolist())
                all_labels.extend(batch['labels'].cpu().tolist())

        val_m = compute_metrics(all_labels, all_preds)
        avg_val_loss = val_loss / len(val_loader)

        # ── Logging ──
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_f1'].append(train_m['f1_macro'])
        history['val_f1'].append(val_m['f1_macro'])
        history['train_acc'].append(train_m['accuracy'])
        history['val_acc'].append(val_m['accuracy'])

        gap = generalization_gap(train_m['f1_macro'], val_m['f1_macro'])
        if verbose:
            print(f'  Époque {epoch}/{num_epochs} — '
                  f'loss={avg_train_loss:.4f}/{avg_val_loss:.4f} | '
                  f'F1={train_m["f1_macro"]:.4f}/{val_m["f1_macro"]:.4f} | '
                  f'gap={gap["gap"]:.4f}')

        # ── Early stopping ──
        if val_m['f1_macro'] > best_val_f1:
            best_val_f1 = val_m['f1_macro']
            best_state  = copy.deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= early_stopping_patience:
                if verbose: print(f'  ⛔ Early stopping à l\'époque {epoch}')
                break

    if best_state: model.load_state_dict(best_state)
    elapsed = time.time() - t_start
    if verbose: print(f'  ✅ Terminé en {elapsed:.0f}s | Meilleur F1-val : {best_val_f1:.4f}')

    return {'best_val_f1': best_val_f1, 'history': history, 'time_s': elapsed}


def evaluate(model, dataset, device=DEVICE, batch_size=32, verbose=False):
    """Évalue le modèle sur un dataset. Retourne les métriques."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    total_loss, all_labels, all_preds = 0., [], []
    with torch.no_grad():
        for batch in loader:
            batch   = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()
            all_preds.extend(outputs.logits.argmax(-1).cpu().tolist())
            all_labels.extend(batch['labels'].cpu().tolist())
    metrics = compute_metrics(all_labels, all_preds)
    metrics['loss'] = total_loss / len(loader)
    if verbose:
        print(classification_report(all_labels, all_preds,
                                    target_names=['Négatif','Positif'], zero_division=0))
    return metrics

print('✅ Fonctions d\'entraînement définies')

---
## 4. Baseline — Hyperparamètres par Défaut

In [ ]:
# ── 4.1 Entraînement baseline ─────────────────────────────────────────────
print('Entraînement BASELINE (wd=1e-4, dropout=0.1, lr=2e-5)...')
print('─'*60)

baseline_model = load_camembert(dropout=0.1)
baseline_result = train_model(
    baseline_model, train_ds, val_ds,
    lr=2e-5, weight_decay=1e-4,
    batch_size=16, grad_accum=2,
    num_epochs=3, verbose=True,
)

baseline_test = evaluate(baseline_model, test_ds)
print(f'\n── Résultats baseline ──')
print(f'  F1-score (test)  : {baseline_test["f1_macro"]:.4f}')
print(f'  Accuracy (test)  : {baseline_test["accuracy"]:.4f}')

gap_b = generalization_gap(
    max(baseline_result['history']['train_f1']),
    baseline_result['best_val_f1']
)
print(f'  Gap (train-val)  : {gap_b["gap"]:.4f} ({gap_b["gap_pct"]:.1f}%)')

In [ ]:
# ── 4.2 Courbes baseline ───────────────────────────────────────────────────
hist = baseline_result['history']
epochs = range(1, len(hist['train_f1'])+1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, hist['train_f1'], 'o-', color=PALETTE[0], lw=2.5, label='Train')
axes[0].plot(epochs, hist['val_f1'],   's--', color=PALETTE[2], lw=2.5, label='Validation')
axes[0].set_xlabel('Époque'); axes[0].set_ylabel('F1-score macro')
axes[0].set_title('F1-score macro par époque (Baseline)'); axes[0].legend()

axes[1].plot(epochs, hist['train_loss'], 'o-', color=PALETTE[0], lw=2.5, label='Train')
axes[1].plot(epochs, hist['val_loss'],   's--', color=PALETTE[2], lw=2.5, label='Validation')
axes[1].set_xlabel('Époque'); axes[1].set_ylabel('Cross-entropy loss')
axes[1].set_title('Loss par époque (Baseline)'); axes[1].legend()

fig.suptitle('CamemBERT Baseline — wd=1e-4, dropout=0.1', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'baseline_curves.png', bbox_inches='tight')
plt.show()

---
## 5. Grid Search P02 — Weight Decay × Dropout

In [ ]:
# ── 5.1 Grid search (12 combinaisons) ────────────────────────────────────
# ⚠️  Chaque trial prend ~5-15 min sur CPU.
# Pour un test rapide, réduire N_TRAIN_PER_CLASS ou num_epochs ci-dessous.

GRID_NUM_EPOCHS = 2   # Réduit pour le grid (Optuna affinera ensuite)
GRID_LR         = 2e-5  # Fixé pour isoler weight_decay et dropout

grid_results = []
grid_histories = {}

total = len(WEIGHT_DECAY_GRID) * len(DROPOUT_GRID)
current = 0

for wd in WEIGHT_DECAY_GRID:
    for dp in DROPOUT_GRID:
        current += 1
        label = f'wd={wd:.0e}_dp={dp}'
        print(f'\n[{current}/{total}] weight_decay={wd:.0e} | dropout={dp}')
        print('─'*55)

        model = load_camembert(dropout=dp)
        result = train_model(
            model, train_ds, val_ds,
            lr=GRID_LR, weight_decay=wd,
            batch_size=16, grad_accum=2,
            num_epochs=GRID_NUM_EPOCHS,
            early_stopping_patience=2,
            verbose=True,
        )

        hist = result['history']
        train_f1 = max(hist['train_f1']) if hist['train_f1'] else 0.
        val_f1   = result['best_val_f1']

        grid_results.append({
            'weight_decay': wd,
            'dropout':      dp,
            'train_f1':     train_f1,
            'val_f1':       val_f1,
            'gap':          train_f1 - val_f1,
            'time_s':       result['time_s'],
        })
        grid_histories[label] = hist
        del model

df_grid = pd.DataFrame(grid_results)
df_grid.to_csv(RESULTS_DIR / 'grid_p02_results.csv', index=False)
print('\n✅ Grid search terminé')
print(df_grid.sort_values('val_f1', ascending=False).to_string(index=False))

In [ ]:
# ── 5.2 Heatmaps Grid P02 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wd_labels = [f'{w:.0e}' for w in sorted(df_grid['weight_decay'].unique())]
dp_labels = [f'{d:.1f}' for d in sorted(df_grid['dropout'].unique())]

# Heatmap gap (rouge = sur-apprentissage)
pivot_gap = df_grid.pivot(index='weight_decay', columns='dropout', values='gap')
sns.heatmap(pivot_gap, ax=axes[0], cmap='Reds', annot=True, fmt='.3f',
            linewidths=0.8, cbar_kws={'label': 'gap'},
            xticklabels=dp_labels, yticklabels=wd_labels)
axes[0].set_title('Gap de généralisation (F1_train − F1_val)\n← plus faible = mieux', fontsize=11)
axes[0].set_xlabel('Dropout', fontsize=11)
axes[0].set_ylabel('Weight Decay', fontsize=11)

# Heatmap F1-val (vert = bon)
pivot_f1 = df_grid.pivot(index='weight_decay', columns='dropout', values='val_f1')
sns.heatmap(pivot_f1, ax=axes[1], cmap='Greens', annot=True, fmt='.3f',
            linewidths=0.8, cbar_kws={'label': 'F1 macro'},
            xticklabels=dp_labels, yticklabels=wd_labels)
axes[1].set_title('F1-score macro (validation)\n← plus élevé = mieux', fontsize=11)
axes[1].set_xlabel('Dropout', fontsize=11)
axes[1].set_ylabel('Weight Decay', fontsize=11)

fig.suptitle('Protocole P02 — Grid Search : Régularisation × Généralisation\n'
             'CamemBERT-base | Allociné | G10', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'heatmap_p02.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.3 Effet isolé du dropout (wd fixé au meilleur) ─────────────────────
best_wd = df_grid.groupby('weight_decay')['val_f1'].mean().idxmax()
df_dp = df_grid[df_grid['weight_decay'] == best_wd].sort_values('dropout')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

x = df_dp['dropout'].values
axes[0].bar(x, df_dp['train_f1'], width=0.04, label='Train F1', color=PALETTE[0], alpha=0.8)
axes[0].bar(x+0.005, df_dp['val_f1'], width=0.04, label='Val F1', color=PALETTE[3], alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels([str(d) for d in x])
axes[0].set_xlabel('Dropout', fontsize=11); axes[0].set_ylabel('F1-score macro', fontsize=11)
axes[0].set_title(f'Effet du dropout (wd={best_wd:.0e})', fontsize=12); axes[0].legend()

axes[1].plot(x, df_dp['gap'], 'o-', color=PALETTE[5], lw=2.5, ms=8)
for xi, yi, label in zip(x, df_dp['gap'], df_dp['dropout']):
    axes[1].annotate(f'{yi:.3f}', (xi, yi), xytext=(0, 8), textcoords='offset points', ha='center')
axes[1].set_xticks(x); axes[1].set_xticklabels([str(d) for d in x])
axes[1].set_xlabel('Dropout', fontsize=11); axes[1].set_ylabel('Gap (F1_train − F1_val)', fontsize=11)
axes[1].set_title(f'Effet du dropout sur le gap (wd={best_wd:.0e})', fontsize=12)

fig.suptitle('Analyse de l\'effet du Dropout — G10 (P02)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dropout_effect.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4 Courbes de convergence comparatives ───────────────────────────────
# Comparer dropout=0.0, 0.1, 0.3 pour le meilleur weight_decay
configs_to_plot = {
    f'dropout=0.0 (wd={best_wd:.0e})': grid_histories.get(f'wd={best_wd:.0e}_dp=0.0', {}),
    f'dropout=0.1 (wd={best_wd:.0e})': grid_histories.get(f'wd={best_wd:.0e}_dp=0.1', {}),
    f'dropout=0.3 (wd={best_wd:.0e})': grid_histories.get(f'wd={best_wd:.0e}_dp=0.3', {}),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, (lbl, hist) in enumerate(configs_to_plot.items()):
    if not hist or not hist.get('train_f1'): continue
    color = PALETTE[i*2]
    ep = range(1, len(hist['train_f1'])+1)
    axes[0].plot(ep, hist['train_f1'], color=color, lw=2, ls='-',  label=f'{lbl} (train)')
    axes[0].plot(ep, hist['val_f1'],   color=color, lw=2, ls='--', label=f'{lbl} (val)')
    axes[1].plot(ep, hist['train_loss'], color=color, lw=2, ls='-')
    axes[1].plot(ep, hist['val_loss'],   color=color, lw=2, ls='--')

axes[0].set_xlabel('Époque'); axes[0].set_ylabel('F1-score macro')
axes[0].set_title('F1-score — effet du dropout'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('Époque'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss — effet du dropout')
legend_elts = [Line2D([0],[0],color='gray',lw=2,ls='-',label='Train'),
               Line2D([0],[0],color='gray',lw=2,ls='--',label='Val')]
axes[1].legend(handles=legend_elts)

fig.suptitle('Courbes de convergence par dropout — G10 (P02)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'convergence_dropout.png', bbox_inches='tight')
plt.show()

---
## 6. Optimisation Bayésienne — Optuna (TPE)

In [ ]:
# ── 6.1 Définition de la fonction objectif ────────────────────────────────
# Sous-ensemble réduit pour accélérer chaque trial Optuna
optuna_train_ds = AllocinéDataset(
    balanced_subsample(raw_dataset['train'], n_per_class=250, seed=SEED),
    tokenizer
)
optuna_val_ds = AllocinéDataset(
    balanced_subsample(raw_dataset['validation'], n_per_class=100, seed=SEED),
    tokenizer
)
print(f'Datasets Optuna : {len(optuna_train_ds)} train | {len(optuna_val_ds)} val')

def objective(trial):
    """
    Objectif Optuna : maximiser F1-score macro sur validation.
    Variables explorées :
      - weight_decay (catégoriel : protocole P02)
      - dropout      (catégoriel : protocole P02)
      - learning_rate (continu log-scale)
    """
    wd  = trial.suggest_categorical('weight_decay', [1e-5, 1e-4, 1e-3, 1e-2])
    dp  = trial.suggest_categorical('dropout',      [0.0, 0.1, 0.3])
    lr  = trial.suggest_float('learning_rate', 1e-6, 5e-4, log=True)

    model  = load_camembert(dropout=dp)
    result = train_model(
        model, optuna_train_ds, optuna_val_ds,
        lr=lr, weight_decay=wd,
        batch_size=16, grad_accum=2,
        num_epochs=2,
        early_stopping_patience=2,
        verbose=False,
    )
    del model

    val_f1 = result['best_val_f1']

    # Stocker métriques complémentaires pour analyse
    hist     = result['history']
    train_f1 = max(hist['train_f1']) if hist['train_f1'] else 0.
    trial.set_user_attr('train_f1', train_f1)
    trial.set_user_attr('gap',      train_f1 - val_f1)
    trial.set_user_attr('time_s',   result['time_s'])

    # Pruning si insuffisant
    if val_f1 < 0.55:
        raise optuna.TrialPruned()

    print(f'  Trial #{trial.number:02d} | wd={wd:.0e} dp={dp:.2f} lr={lr:.2e} '
          f'→ F1_val={val_f1:.4f} (gap={train_f1-val_f1:.4f})')
    return val_f1

print('✅ Fonction objectif définie')

In [ ]:
# ── 6.2 Lancement de l'étude Optuna ──────────────────────────────────────
# ⚠️  N_TRIALS = 20 est idéal. Réduire à 5-10 pour un test rapide.
N_TRIALS = 20

sampler = optuna.samplers.TPESampler(seed=SEED)          # Bayésien (TPE)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=3)

study = optuna.create_study(
    study_name='g10_p02_regularisation',
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    storage=f'sqlite:///{RESULTS_DIR}/optuna.db',
    load_if_exists=True,
)

print(f'Démarrage Optuna — {N_TRIALS} trials (TPE Bayésien)...')
print('─'*60)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

print('\n' + '='*60)
print('  RÉSULTATS OPTUNA')
print('='*60)
print(f'  Meilleur F1-val  : {study.best_value:.4f}')
print(f'  Meilleurs params :')
for k, v in study.best_params.items():
    print(f'    {k:20s} = {v}')

# Sauvegarde
with open(RESULTS_DIR / 'optuna_study.pkl', 'wb') as f:
    pickle.dump(study, f)
best_params = {'best_value': study.best_value, 'best_params': study.best_params}
with open(RESULTS_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print('\n✅ Étude sauvegardée')

In [ ]:
# ── 6.3 Tableau des meilleurs trials ─────────────────────────────────────
df_trials = study.trials_dataframe()
df_trials = df_trials[df_trials['state'] == 'COMPLETE'].sort_values('value', ascending=False)

cols = ['number', 'value', 'params_weight_decay', 'params_dropout', 'params_learning_rate',
        'user_attrs_train_f1', 'user_attrs_gap']
cols = [c for c in cols if c in df_trials.columns]
print('Top 10 trials :')
print(df_trials[cols].head(10).to_string(index=False))

In [ ]:
# ── 6.4 Visualisations Optuna ─────────────────────────────────────────────
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) Historique de l'optimisation
trial_nums = [t.number for t in completed]
f1_vals    = [t.value  for t in completed]
best_so_far = np.maximum.accumulate(f1_vals)
axes[0].scatter(trial_nums, f1_vals, alpha=0.6, color=PALETTE[1], s=50, zorder=3)
axes[0].plot(trial_nums, best_so_far, color=PALETTE[5], lw=2.5, label='Meilleur cumulé')
axes[0].set_xlabel('Trial #'); axes[0].set_ylabel('F1-score macro (val)')
axes[0].set_title('Convergence Optuna (TPE)'); axes[0].legend()

# (b) Distribution des F1 par dropout
dropout_vals  = [t.params.get('dropout', None)  for t in completed]
f1_vals_clean = [t.value for t in completed if t.params.get('dropout') is not None]
for dp_val in sorted(set(dropout_vals)):
    idxs = [i for i, d in enumerate(dropout_vals) if d == dp_val]
    vals = [f1_vals[i] for i in idxs]
    axes[1].scatter([dp_val]*len(vals), vals, alpha=0.7, s=60,
                    label=f'dropout={dp_val}', zorder=3)
axes[1].set_xlabel('Dropout'); axes[1].set_ylabel('F1-score macro (val)')
axes[1].set_title('F1-val par valeur de dropout'); axes[1].legend(fontsize=9)

# (c) Distribution des F1 par weight_decay
wd_vals = [t.params.get('weight_decay', None) for t in completed]
for wd_val in sorted(set(v for v in wd_vals if v is not None)):
    idxs = [i for i, w in enumerate(wd_vals) if w == wd_val]
    vals = [f1_vals[i] for i in idxs]
    axes[2].scatter([np.log10(wd_val)]*len(vals), vals, alpha=0.7, s=60,
                    label=f'wd={wd_val:.0e}', zorder=3)
axes[2].set_xlabel('log10(weight_decay)'); axes[2].set_ylabel('F1-score macro (val)')
axes[2].set_title('F1-val par weight_decay'); axes[2].legend(fontsize=9)

fig.suptitle('Analyse de l\'étude Optuna — G10 (P02)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'optuna_analysis.png', bbox_inches='tight')
plt.show()

---
## 7. Analyse du Loss Landscape & Sharpness

In [ ]:
# ── 7.1 Calcul du loss landscape 1D ──────────────────────────────────────
def compute_loss_landscape_1d(
    model, dataset, device=DEVICE,
    n_points=8, epsilon=0.05, n_samples=50
):
    """
    Perturbe le modèle dans une direction aléatoire normalisée
    et mesure la loss pour α ∈ [-ε, +ε].

    Filter normalization (Li et al. 2018) : la direction est
    mise à l'échelle filtre par filtre pour que la perturbation
    soit comparable indépendamment de la taille des paramètres.
    """
    model.eval()
    orig_params = [p.clone().detach() for p in model.parameters()]

    # Direction filtre-normalisée
    direction = []
    for p in orig_params:
        d = torch.randn_like(p)
        # Filter normalization : ||d||_F = ||p||_F par filtre
        if p.dim() >= 2:
            for i in range(p.shape[0]):
                n_p = p[i].norm()
                n_d = d[i].norm()
                if n_d > 1e-8 and n_p > 1e-8:
                    d[i] = d[i] * (n_p / n_d)
        direction.append(d)

    loader = DataLoader(dataset, batch_size=16, shuffle=False)
    n_batches = max(1, n_samples // 16)
    alphas = np.linspace(-epsilon, epsilon, n_points)
    losses = []

    for alpha in alphas:
        # Perturber
        for p, p0, d in zip(model.parameters(), orig_params, direction):
            p.data = p0 + float(alpha) * d.to(p0.device)
        # Évaluer
        batch_losses = []
        with torch.no_grad():
            for i, batch in enumerate(loader):
                if i >= n_batches: break
                batch = {k: v.to(device) for k, v in batch.items()}
                batch_losses.append(model(**batch).loss.item())
        losses.append(float(np.mean(batch_losses)))

    # Restaurer θ*
    for p, p0 in zip(model.parameters(), orig_params):
        p.data = p0.to(p.device)

    return alphas, losses

print('✅ Fonction loss landscape définie')

In [ ]:
# ── 7.2 Calcul pour 3 dropouts (weight_decay = meilleur du grid) ─────────
# Entraîner 3 modèles avec dropout = 0.0, 0.1, 0.3
landscape_results = {}   # label → (alphas, losses)
sharpness_data    = []   # pour le scatter plot

LANDSCAPE_WD = best_wd   # weight_decay optimal trouvé dans le grid
LANDSCAPE_LR = study.best_params.get('learning_rate', 2e-5)

for dp in [0.0, 0.1, 0.3]:
    lbl = f'dropout={dp:.1f}'
    print(f'\nEntraînement pour loss landscape : {lbl}...')

    model = load_camembert(dropout=dp)
    result = train_model(
        model, train_ds, val_ds,
        lr=LANDSCAPE_LR, weight_decay=LANDSCAPE_WD,
        batch_size=16, grad_accum=2, num_epochs=3,
        verbose=False,
    )

    print(f'  F1-val={result["best_val_f1"]:.4f} | Calcul du landscape...')
    alphas, losses = compute_loss_landscape_1d(model, val_ds)
    landscape_results[lbl] = (alphas, losses)

    base_loss = losses[len(losses)//2]
    sharpness = compute_sharpness(base_loss, losses)
    sharpness_data.append({
        'label': lbl,
        'sharpness': sharpness,
        'val_f1': result['best_val_f1'],
        'dropout': dp,
    })
    print(f'  Sharpness={sharpness:.5f}')
    del model

# Sauvegarde
with open(RESULTS_DIR / 'sharpness_results.json', 'w') as f:
    json.dump(sharpness_data, f, indent=2)
print('\n✅ Loss landscapes calculés')

In [ ]:
# ── 7.3 Visualisation du loss landscape ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

styles = [
    {'color': PALETTE[0], 'ls': '-',  'marker': 'o'},
    {'color': PALETTE[3], 'ls': '--', 'marker': 's'},
    {'color': PALETTE[5], 'ls': ':',  'marker': '^'},
]

for i, (lbl, (alphas, losses)) in enumerate(landscape_results.items()):
    s = styles[i]
    base_loss = losses[len(losses)//2]
    sharpness = compute_sharpness(base_loss, losses)
    axes[0].plot(alphas, losses, color=s['color'], lw=2.5, ls=s['ls'],
                 marker=s['marker'], ms=6, markerfacecolor='white',
                 label=f'{lbl}  (sharpness={sharpness:.4f})')

axes[0].axvline(0, color='gray', lw=0.8, ls='--', alpha=0.5, label='θ* (convergence)')
axes[0].set_xlabel('Direction de perturbation (α)', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('Loss landscape 1D\n(perturbation filtre-normalisée)', fontsize=12)
axes[0].legend(fontsize=9)

# Scatter sharpness vs F1-val
for i, d in enumerate(sharpness_data):
    axes[1].scatter(d['sharpness'], d['val_f1'],
                    color=styles[i]['color'], s=150, zorder=5,
                    edgecolors='white', linewidths=1.5)
    axes[1].annotate(d['label'], (d['sharpness'], d['val_f1']),
                     xytext=(6, 4), textcoords='offset points', fontsize=10)

# Tendance
if len(sharpness_data) >= 2:
    xs = [d['sharpness'] for d in sharpness_data]
    ys = [d['val_f1']    for d in sharpness_data]
    z  = np.polyfit(xs, ys, 1)
    xr = np.linspace(min(xs)*0.9, max(xs)*1.1, 50)
    axes[1].plot(xr, np.poly1d(z)(xr), 'k--', lw=1.5, alpha=0.5, label='Tendance')
    axes[1].legend(fontsize=10)

axes[1].set_xlabel('Sharpness du minimum', fontsize=11)
axes[1].set_ylabel('F1-score macro (validation)', fontsize=11)
axes[1].set_title('Sharpness ↔ Généralisation\nMinima plats → meilleure généralisation', fontsize=12)

fig.suptitle('Analyse du Loss Landscape — CamemBERT / Allociné (G10 — P02)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'loss_landscape.png', bbox_inches='tight')
plt.show()

---
## 8. Entraînement Final — Meilleurs Hyperparamètres

In [ ]:
# ── 8.1 Récupérer les meilleurs params ────────────────────────────────────
best = study.best_params
BEST_LR  = best['learning_rate']
BEST_WD  = best['weight_decay']
BEST_DP  = best['dropout']

print('Hyperparamètres optimaux (Optuna) :')
print(f'  learning_rate = {BEST_LR:.2e}')
print(f'  weight_decay  = {BEST_WD:.1e}')
print(f'  dropout       = {BEST_DP}')
print(f'\nDémarrage de l\'entraînement final...')
print('─'*60)

final_model = load_camembert(dropout=BEST_DP)
final_result = train_model(
    final_model, train_ds, val_ds,
    lr=BEST_LR, weight_decay=BEST_WD,
    batch_size=16, grad_accum=2,
    num_epochs=5,
    early_stopping_patience=3,
    verbose=True,
)

# Sauvegarde du modèle final
torch.save(final_model.state_dict(), RESULTS_DIR / 'best_model.pt')
print('\n✅ Modèle final sauvegardé')

In [ ]:
# ── 8.2 Évaluation sur le jeu de TEST ─────────────────────────────────────
print('Évaluation sur le jeu de test...')
test_metrics = evaluate(final_model, test_ds, verbose=True)

baseline_test_f1 = baseline_test['f1_macro']
gain = (test_metrics['f1_macro'] - baseline_test_f1) / baseline_test_f1 * 100

print(f'\n{'='*55}')
print('  RÉSULTATS FINAUX — G10 (P02)')
print('='*55)
print(f'  F1-score test (baseline)  : {baseline_test_f1:.4f}')
print(f'  F1-score test (optimisé)  : {test_metrics["f1_macro"]:.4f}')
print(f'  Gain                      : +{gain:.1f}%')
print(f'  Accuracy test             : {test_metrics["accuracy"]:.4f}')
gap_final = generalization_gap(
    max(final_result['history']['train_f1']),
    final_result['best_val_f1']
)
print(f'  Gap final (train-val)     : {gap_final["gap"]:.4f} ({gap_final["gap_pct"]:.1f}%)')
print('='*55)

In [ ]:
# ── 8.3 Courbes d'entraînement final ─────────────────────────────────────
hist_f = final_result['history']
hist_b = baseline_result['history']
ep_f = range(1, len(hist_f['train_f1'])+1)
ep_b = range(1, len(hist_b['train_f1'])+1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# F1
axes[0].plot(ep_b, hist_b['val_f1'], 'o--', color=PALETTE[1], lw=2, label='Baseline (val)', alpha=0.7)
axes[0].plot(ep_f, hist_f['train_f1'], 'o-', color=PALETTE[0], lw=2.5, label='Final (train)')
axes[0].plot(ep_f, hist_f['val_f1'],   's--', color=PALETTE[3], lw=2.5, label='Final (val)')
axes[0].set_xlabel('Époque'); axes[0].set_ylabel('F1-score macro')
axes[0].set_title('F1-score — Modèle final vs Baseline'); axes[0].legend(fontsize=9)

# Loss
axes[1].plot(ep_b, hist_b['val_loss'], 'o--', color=PALETTE[1], lw=2, label='Baseline (val)', alpha=0.7)
axes[1].plot(ep_f, hist_f['train_loss'], 'o-', color=PALETTE[0], lw=2.5, label='Final (train)')
axes[1].plot(ep_f, hist_f['val_loss'],   's--', color=PALETTE[3], lw=2.5, label='Final (val)')
axes[1].set_xlabel('Époque'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss — Modèle final vs Baseline'); axes[1].legend(fontsize=9)

fig.suptitle(f'Entraînement Final — wd={BEST_WD:.0e}, dropout={BEST_DP}, lr={BEST_LR:.1e}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_training_curves.png', bbox_inches='tight')
plt.show()

---
## 9. Synthèse & Visualisations Finales

In [ ]:
# ── 9.1 Tableau comparatif complet ────────────────────────────────────────
summary_data = {
    'Configuration': [
        'Baseline (wd=1e-4, dp=0.1)',
        f'Grid Optimal (wd={best_wd:.0e}, dp=?)'
        if df_grid is not None else 'Grid Optimal',
        f'Optuna Optimal (wd={BEST_WD:.0e}, dp={BEST_DP}, lr={BEST_LR:.1e})',
    ],
    'F1 (val)': [
        baseline_result['best_val_f1'],
        df_grid.sort_values('val_f1').iloc[-1]['val_f1'] if df_grid is not None else float('nan'),
        final_result['best_val_f1'],
    ],
    'F1 (test)': [
        baseline_test['f1_macro'],
        float('nan'),
        test_metrics['f1_macro'],
    ],
    'Gap (train-val)': [
        generalization_gap(max(baseline_result['history']['train_f1']),
                           baseline_result['best_val_f1'])['gap'],
        df_grid.sort_values('val_f1').iloc[-1]['gap'] if df_grid is not None else float('nan'),
        gap_final['gap'],
    ],
}

df_summary = pd.DataFrame(summary_data)
print('\n── Tableau de synthèse ──')
print(df_summary.to_string(index=False, float_format='{:.4f}'.format))
df_summary.to_csv(RESULTS_DIR / 'summary.csv', index=False)

In [ ]:
# ── 9.2 Figure de synthèse finale ─────────────────────────────────────────
fig = plt.figure(figsize=(17, 12))
gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# (A) Heatmap F1-val
ax_a = fig.add_subplot(gs[0, 0])
if df_grid is not None:
    pivot = df_grid.pivot(index='weight_decay', columns='dropout', values='val_f1')
    sns.heatmap(pivot, ax=ax_a, cmap='YlGn', annot=True, fmt='.3f',
                linewidths=0.5, cbar=False,
                xticklabels=[f'{d:.1f}' for d in sorted(df_grid['dropout'].unique())],
                yticklabels=[f'{w:.0e}' for w in sorted(df_grid['weight_decay'].unique())])
ax_a.set_title('(A) F1-val : grid P02', fontsize=10, fontweight='bold')
ax_a.set_xlabel('Dropout', fontsize=9); ax_a.set_ylabel('Weight Decay', fontsize=9)

# (B) Heatmap gap
ax_b = fig.add_subplot(gs[0, 1])
if df_grid is not None:
    pivot_g = df_grid.pivot(index='weight_decay', columns='dropout', values='gap')
    sns.heatmap(pivot_g, ax=ax_b, cmap='Reds', annot=True, fmt='.3f',
                linewidths=0.5, cbar=False,
                xticklabels=[f'{d:.1f}' for d in sorted(df_grid['dropout'].unique())],
                yticklabels=[f'{w:.0e}' for w in sorted(df_grid['weight_decay'].unique())])
ax_b.set_title('(B) Gap génér. : grid P02', fontsize=10, fontweight='bold')
ax_b.set_xlabel('Dropout', fontsize=9); ax_b.set_ylabel('', fontsize=9)

# (C) Optuna convergence
ax_c = fig.add_subplot(gs[0, 2])
f1_vals_optuna = [t.value for t in completed]
ax_c.scatter(range(len(f1_vals_optuna)), f1_vals_optuna, alpha=0.6, color=PALETTE[1], s=40)
ax_c.plot(range(len(f1_vals_optuna)), np.maximum.accumulate(f1_vals_optuna),
          color=PALETTE[5], lw=2.5)
ax_c.set_xlabel('Trial'); ax_c.set_ylabel('F1-val')
ax_c.set_title('(C) Convergence Optuna', fontsize=10, fontweight='bold')

# (D) Loss landscape
ax_d = fig.add_subplot(gs[1, :2])
for i, (lbl, (alphas, losses)) in enumerate(landscape_results.items()):
    s = styles[i]
    sharpness = compute_sharpness(losses[len(losses)//2], losses)
    ax_d.plot(alphas, losses, color=s['color'], lw=2.5, ls=s['ls'],
              marker=s['marker'], ms=5, markerfacecolor='white',
              label=f'{lbl}  (S={sharpness:.4f})')
ax_d.axvline(0, color='gray', lw=0.8, ls='--', alpha=0.5)
ax_d.set_xlabel('Direction de perturbation α', fontsize=10)
ax_d.set_ylabel('Loss', fontsize=10)
ax_d.set_title('(D) Loss Landscape 1D — effet du dropout', fontsize=10, fontweight='bold')
ax_d.legend(fontsize=8)

# (E) Sharpness vs F1
ax_e = fig.add_subplot(gs[1, 2])
for i, d in enumerate(sharpness_data):
    ax_e.scatter(d['sharpness'], d['val_f1'], color=styles[i]['color'], s=120,
                 zorder=5, edgecolors='white', lw=1.5)
    ax_e.annotate(d['label'], (d['sharpness'], d['val_f1']),
                  xytext=(5, 4), textcoords='offset points', fontsize=8)
ax_e.set_xlabel('Sharpness', fontsize=10); ax_e.set_ylabel('F1-val', fontsize=10)
ax_e.set_title('(E) Sharpness ↔ Généralisation', fontsize=10, fontweight='bold')

# (F) Courbes finales
ax_f = fig.add_subplot(gs[2, :])
ep_f = range(1, len(hist_f['train_f1'])+1)
ax_f.plot(ep_f, hist_f['train_f1'], 'o-', color=PALETTE[0], lw=2.5, label='Final — Train F1')
ax_f.plot(ep_f, hist_f['val_f1'],   's--', color=PALETTE[3], lw=2.5, label='Final — Val F1')
ax_f.axhline(baseline_test['f1_macro'], color='gray', lw=1.5, ls=':',
             label=f'Baseline test F1 = {baseline_test["f1_macro"]:.3f}')
ax_f.axhline(test_metrics['f1_macro'],  color=PALETTE[5], lw=1.5, ls=':',
             label=f'Final test F1 = {test_metrics["f1_macro"]:.3f}')
ax_f.fill_between(ep_f, hist_f['train_f1'], hist_f['val_f1'],
                  alpha=0.1, color=PALETTE[0], label='Gap train/val')
ax_f.set_xlabel('Époque', fontsize=10); ax_f.set_ylabel('F1-score macro', fontsize=10)
ax_f.set_title('(F) Entraînement final — Meilleurs hyperparamètres Optuna', fontsize=10, fontweight='bold')
ax_f.legend(fontsize=9, ncol=2)

fig.suptitle(
    'G10 — Synthèse : Régularisation & Généralisation de CamemBERT (P02)\n'
    f'Dataset : Allociné (D05) | Modèle : CamemBERT-base (M04) | Optuna TPE',
    fontsize=14, fontweight='bold'
)
plt.savefig(FIGURES_DIR / 'synthese_finale.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figure de synthèse sauvegardée')

---
## 10. Conclusions

In [ ]:
# ── 10.1 Rapport de conclusion automatique ────────────────────────────────
print('=' * 65)
print('  CONCLUSIONS — G10 : Régularisation & Généralisation (P02)')
print('=' * 65)

print(f"""
PROTOCOLE
  Modèle    : CamemBERT-base (110M paramètres)
  Dataset   : Allociné (D05) — {len(train_samples)} train, {len(val_samples)} val, {len(test_samples)} test
  Méthode   : Optuna TPE (Bayésien) + Grid Search exhaustif P02
  Métriq.   : F1-score macro

RÉSULTATS
  Baseline  : F1-test = {baseline_test['f1_macro']:.4f}  (wd=1e-4, dropout=0.1)
  Optimisé  : F1-test = {test_metrics['f1_macro']:.4f}  (wd={BEST_WD:.0e}, dropout={BEST_DP}, lr={BEST_LR:.1e})
  Gain      : +{(test_metrics['f1_macro'] - baseline_test['f1_macro'])/baseline_test['f1_macro']*100:.1f}%

EFFET DU WEIGHT DECAY
  → Réduit le sur-apprentissage en pénalisant les grands poids
  → Valeurs trop fortes (1e-2) : sous-apprentissage
  → Valeur optimale : {BEST_WD:.0e}

EFFET DU DROPOUT
  → Dropout = 0.0 : F1 train élevé mais gap train/val important
  → Dropout = 0.3 : régularisation forte mais convergence plus lente
  → Valeur optimale : {BEST_DP}

LOSS LANDSCAPE
  → Minima plats (sharpness faible) corrélés à une meilleure généralisation
  → Le dropout favorise des minima plus plats
  → Cohérent avec Hochreiter & Schmidhuber (1997) et Keskar et al. (2017)

ADAPTATION CPU
  → Gradient accumulation (batch effectif = 32)
  → Sous-échantillonnage équilibré ({N_TRAIN_PER_CLASS}/classe)
  → Early stopping (patience=3)
  → Loss landscape 1D allégé (8 points, ε=0.05)
""")
print('=' * 65)

In [ ]:
# ── 10.2 Fichiers produits ────────────────────────────────────────────────
print('\nFichiers produits :')
for f in sorted(RESULTS_DIR.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        unit = 'Ko' if size < 1e6 else 'Mo'
        print(f'  {str(f.relative_to(RESULTS_DIR)):45s}  {size/1e3 if size < 1e6 else size/1e6:.1f} {unit}')